In [4]:
%pip install shap

Note: you may need to restart the kernel to use updated packages.


In [5]:
import shap
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForMaskedLM

# 1. 加载模型和分词器
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)
model.eval()

# 2. 定义你想分析的单条数据
n1, n2 = "virus", "infection"
prompt = f"A {n1} {n2} refers to an {n2} that <mask> a {n1}."
# 假设你想分析模型为什么预测 "causes" (注意 RoBERTa 中词汇前通常有个空格 Ġ)
target_word = " causes" 
target_id = tokenizer.convert_tokens_to_ids(target_word)

# 3. 构造自定义预测函数
def mlm_predict_score(texts):
    """
    输入: 一组被 SHAP 扰动过的文本列表 (例如: ["A virus infection refers...", "A virus [PAD] refers..."])
    输出: 在 <mask> 位置，预测 target_word 的 Logit 分数
    """
    scores = []
    for text in texts:
        inputs = tokenizer(text, return_tensors="pt")
        
        # 找到 <mask> token 的位置索引
        # RoBERTa 的 mask token id 是 tokenizer.mask_token_id (通常是 50264)
        mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
        
        # 如果文本被扰动得连 mask 都不见了，给个极低分
        if len(mask_token_index) == 0:
            scores.append(-100.0)
            continue
            
        with torch.no_grad():
            outputs = model(**inputs)
        
        # 获取 mask 位置的所有词汇的 logits
        mask_logits = outputs.logits[0, mask_token_index, :]
        
        # 提取目标词 (如 "causes") 的得分
        target_score = mask_logits[0, target_id].item()
        scores.append(target_score)
        
    return np.array(scores)

# 4. 初始化 SHAP Explainer
# 告诉 SHAP 使用我们的分词器来切分特征，并用 mlm_predict_score 来打分
explainer = shap.Explainer(mlm_predict_score, tokenizer)

# 5. 计算 SHAP 值
# 注意：这里传入的是只包含一个元素的列表
shap_values = explainer([prompt])

# 6. 可视化分析结果
# 这会生成一个基于文本的可视化图表，红色表示正向贡献，蓝色表示负向贡献
shap.plots.text(shap_values[0])

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

RobertaForMaskedLM LOAD REPORT from: roberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  0%|          | 0/182 [00:00<?, ?it/s]

PartitionExplainer explainer: 2it [00:15, 15.95s/it]               


In [8]:
import shap
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForMaskedLM

# 1. 加载模型
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)
model.eval()

# 2. 载入我们失败的案例: 
# Relation: attribute
# ------------------------------------------------------------
# Compound   : incinerator plant
# Prompt     : An incinerator plant refers to a plant that <mask> incinerator.
# Predictions: ['produces', 'uses', 'processes', 'manufactures', 'burns']
# ------------------------------------------------------------

n1, n2 = "incinerator", "plant"
# 这里按照原始 Prompt 运行
prompt = f"An {n1} {n2} refers to a {n2} that <mask> {n1}."

# 3. 定义【真实的正确词】和【模型预测错的词】
# 选 ' features' 作为 attribute 关系的代表词；选 ' produces' 作为模型预测错的词
correct_word = " features" 
wrong_word = " produces" 

correct_id = tokenizer.convert_tokens_to_ids(correct_word)
wrong_id = tokenizer.convert_tokens_to_ids(wrong_word)

def mlm_predict_scores(texts):
    scores_correct = []
    scores_wrong = []
    
    for text in texts:
        inputs = tokenizer(text, return_tensors="pt")
        # 寻找 <mask_token> (RoBERTa 默认是 <mask>)
        mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
        
        if len(mask_token_index) == 0:
            scores_correct.append(-100.0)
            scores_wrong.append(-100.0)
            continue
            
        with torch.no_grad():
            outputs = model(**inputs)
        
        # 提取目标词的分数
        mask_logits = outputs.logits[0, mask_token_index[0], :]
        scores_correct.append(mask_logits[correct_id].item())
        scores_wrong.append(mask_logits[wrong_id].item())
        
    return np.stack([scores_correct, scores_wrong], axis=-1)

# 4. 运行 SHAP
explainer = shap.Explainer(mlm_predict_scores, tokenizer)
shap_values = explainer([prompt])

# ==========================================
# 5. 可视化保存（这次用 Bar plot，更直观比对每个词的权重）
# ==========================================

# 图 1：看看哪些词在“阻止”模型预测 features
plt.figure(figsize=(10, 5))
shap.plots.bar(shap_values[0, :, 0], show=False)
plt.title("Why the model DID NOT predict 'features' (Correct)")
plt.savefig("shap_correct_features.png", bbox_inches="tight")
plt.close()

# 图 2：看看哪些词在“诱导”模型预测 produces
plt.figure(figsize=(10, 5))
shap.plots.bar(shap_values[0, :, 1], show=False)
plt.title("Why the model PREDICTED 'produces' (Wrong)")
plt.savefig("shap_wrong_produces.png", bbox_inches="tight")
plt.close()

print("分析完成！请查看生成的图片：shap_correct_features.png 和 shap_wrong_produces.png")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

RobertaForMaskedLM LOAD REPORT from: roberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  0%|          | 0/210 [00:00<?, ?it/s]

PartitionExplainer explainer: 2it [00:14, 14.77s/it]               


分析完成！请查看生成的图片：shap_correct_features.png 和 shap_wrong_produces.png


In [10]:
import shap
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForMaskedLM

# 1. 加载模型
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)
model.eval()

# 2. 载入我们失败的案例: 
# Relation: attribute
# ------------------------------------------------------------
# Compound   : incinerator plant
# Prompt     : An incinerator plant refers to a plant that <mask> incinerator.
# Predictions: ['produces', 'uses', 'processes', 'manufactures', 'burns']
# ------------------------------------------------------------

n1, n2 = "incinerator", "plant"
# 加上一个 an，让语法变得丝滑
prompt = "An incinerator plant refers to a plant that <mask> an incinerator."

# 3. 定义【真实的正确词】和【模型预测错的词】
# 选 ' features' 作为 attribute 关系的代表词；选 ' produces' 作为模型预测错的词
correct_word = " features" 
wrong_word = " produces" 

correct_id = tokenizer.convert_tokens_to_ids(correct_word)
wrong_id = tokenizer.convert_tokens_to_ids(wrong_word)

def mlm_predict_scores(texts):
    scores_correct = []
    scores_wrong = []
    
    for text in texts:
        inputs = tokenizer(text, return_tensors="pt")
        # 寻找 <mask_token> (RoBERTa 默认是 <mask>)
        mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
        
        if len(mask_token_index) == 0:
            scores_correct.append(-100.0)
            scores_wrong.append(-100.0)
            continue
            
        with torch.no_grad():
            outputs = model(**inputs)
        
        # 提取目标词的分数
        mask_logits = outputs.logits[0, mask_token_index[0], :]
        scores_correct.append(mask_logits[correct_id].item())
        scores_wrong.append(mask_logits[wrong_id].item())
        
    return np.stack([scores_correct, scores_wrong], axis=-1)

# 4. 运行 SHAP
explainer = shap.Explainer(mlm_predict_scores, tokenizer)
shap_values = explainer([prompt])

# ==========================================
# 5. 可视化保存（这次用 Bar plot，更直观比对每个词的权重）
# ==========================================

# 图 1：看看哪些词在“阻止”模型预测 features
plt.figure(figsize=(10, 5))
shap.plots.bar(shap_values[0, :, 0], show=False)
plt.title("Why the model DID NOT predict 'features' (Correct)")
plt.savefig("shap_correct_features_with_an.png", bbox_inches="tight")
plt.close()

# 图 2：看看哪些词在“诱导”模型预测 produces
plt.figure(figsize=(10, 5))
shap.plots.bar(shap_values[0, :, 1], show=False)
plt.title("Why the model PREDICTED 'produces' (Wrong)")
plt.savefig("shap_wrong_produces_with_an.png", bbox_inches="tight")
plt.close()

print("分析完成！请查看生成的图片：shap_correct_features.png 和 shap_wrong_produces.png")


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

RobertaForMaskedLM LOAD REPORT from: roberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  0%|          | 0/240 [00:00<?, ?it/s]

PartitionExplainer explainer: 2it [00:16, 16.61s/it]               


分析完成！请查看生成的图片：shap_correct_features.png 和 shap_wrong_produces.png
